# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarahnjunge/starter-notebooks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Sarahnjunge/starter-notebooks.git
%cd starter-notebooks
!ls data/raw

Cloning into 'starter-notebooks'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 183 (delta 82), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 1.90 MiB | 11.53 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/content/starter-notebooks
content_refresh_anonymized.csv


## 1. Method choice and why


I will start with Logistic Regression because the task has a binary observed label (`down` vs not `down`) and the model is easy to inspect. I will use the predicted probability of `down` to rank pages, because the capstone question is which pages should be reviewed first rather than simply assigning every page a class. I will later compare this with a Random Forest to test whether nonlinear relationships improve the ranking.


In [2]:
# ML-08 setup: define the modeling target and feature vector

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Target: pages observed as "down"
df["target_down"] = (
    df["trend_direction"] == "down"
).astype(int)

# Final feature vector from ML-05
feature_columns = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier"
]

print("Dataset shape:", df.shape)
print("Number of features:", len(feature_columns))

print("\nTarget distribution:")
print(df["target_down"].value_counts().sort_index())

print("\nTarget base rate:")
print(f"{df['target_down'].mean():.3f}")

Dataset shape: (30000, 45)
Number of features: 38

Target distribution:
target_down
0    13738
1    16262
Name: count, dtype: int64

Target base rate:
0.542


## 2. Split design

I will use a grouped-by-client split so that pages from the same client do not appear in both training and test sets. This is more honest than a random row split because the model should be tested on clients it did not use for training. I will keep the test set at 20% of the data and use the same test slice for comparing the ML model with the frozen baseline.


In [3]:
# Create an honest grouped-by-client train/test split

from sklearn.model_selection import GroupShuffleSplit

X = df[feature_columns].copy()
y = df["target_down"].copy()
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nClient overlap:",
      len(train_clients.intersection(test_clients)))

print("\nTraining positive rate:",
      f"{y_train.mean():.3f}")

print("Test positive rate:",
      f"{y_test.mean():.3f}")


Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7

Client overlap: 0

Training positive rate: 0.550
Test positive rate: 0.511


## 3. Train + compare vs my baseline

I will train Logistic Regression on the training clients only, then rank the unseen test pages by predicted probability of decline. I will evaluate Precision@20 and Precision@50.

For a fair comparison, I will also rebuild my frozen baseline on the same test pages and use the same metrics. The overall test-set base rate will be included so I can see whether either approach improves on simply selecting pages at random.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numeric and categorical features
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)


Numeric features: 29
Categorical features: 9

Categorical features:
['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [5]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [6]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

print("Logistic Regression pipeline created successfully.")

Logistic Regression pipeline created successfully.


In [7]:
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [8]:
test_probabilities = logistic_model.predict_proba(X_test)[:, 1]

print("Test probabilities generated successfully.")
print("Number of test predictions:", len(test_probabilities))
print("Minimum probability:", f"{test_probabilities.min():.3f}")
print("Maximum probability:", f"{test_probabilities.max():.3f}")
print("Mean probability:", f"{test_probabilities.mean():.3f}")

Test probabilities generated successfully.
Number of test predictions: 6163
Minimum probability: 0.000
Maximum probability: 1.000
Mean probability: 0.515


In [9]:
model_results = df.iloc[test_idx][
    ["content_id", "client_id"]
].copy()

model_results["actual_down"] = y_test.values
model_results["predicted_probability"] = test_probabilities

model_results = model_results.sort_values(
    "predicted_probability",
    ascending=False
).reset_index(drop=True)

model_results["rank"] = np.arange(1, len(model_results) + 1)

print("Ranked model queue created.")
print("\nTop 10 pages:")
print(
    model_results[
        [
            "rank",
            "content_id",
            "client_id",
            "predicted_probability",
            "actual_down"
        ]
    ].head(10)
)

Ranked model queue created.

Top 10 pages:
   rank            content_id          client_id  predicted_probability  \
0     1  content_2dc625dcdf98  client_4e07408562                    1.0   
1     2  content_6926821766df  client_f369cb89fc                    1.0   
2     3  content_ca17a024f90c  client_4e07408562                    1.0   
3     4  content_007d1f134801  client_4e07408562                    1.0   
4     5  content_c9147cda01f0  client_4e07408562                    1.0   
5     6  content_60a90d0ba16a  client_f369cb89fc                    1.0   
6     7  content_d7175187ff12  client_4e07408562                    1.0   
7     8  content_fca1bf3940c0  client_4e07408562                    1.0   
8     9  content_b138c0b35a91  client_4e07408562                    1.0   
9    10  content_5fe46e04994d  client_4e07408562                    1.0   

   actual_down  
0            1  
1            1  
2            1  
3            1  
4            1  
5            1  
6           

In [10]:
def precision_at_k(results, k):
    top_k = results.head(k)
    return top_k["actual_down"].mean()

p20_logistic = precision_at_k(model_results, 20)
p50_logistic = precision_at_k(model_results, 50)

print("Logistic Regression results")
print("----------------------------")
print(f"Precision@20: {p20_logistic:.3f}")
print(f"Precision@50: {p50_logistic:.3f}")

print("\nPositive pages selected:")
print(f"Top 20: {model_results.head(20)['actual_down'].sum()}/20")
print(f"Top 50: {model_results.head(50)['actual_down'].sum()}/50")

print("\nTest-set base rate:")
print(f"{y_test.mean():.3f}")

Logistic Regression results
----------------------------
Precision@20: 1.000
Precision@50: 1.000

Positive pages selected:
Top 20: 20/20
Top 50: 50/50

Test-set base rate:
0.511


In [11]:
# Rebuild the frozen baseline signals
visible_signal = df["impressions_90d"] > 0

stale_signal = df["days_since_last_update"] > 180

low_ctr_signal = (
    visible_signal &
    (df["ctr"] < df["ctr"].median())
)

weak_engagement_signal = (
    (df["sessions_90d"] > 0) &
    (df["engagement_rate"] < df["engagement_rate"].median())
)

df["baseline_action_score"] = (
    stale_signal.astype(int)
    + low_ctr_signal.astype(int)
    + weak_engagement_signal.astype(int)
)

df["baseline_reason_code"] = np.select(
    [
        stale_signal & low_ctr_signal,
        stale_signal,
        low_ctr_signal,
        weak_engagement_signal
    ],
    [
        "multiple_signals",
        "stale_visible",
        "low_ctr_visible",
        "weak_engagement"
    ],
    default="no_clear_signal"
)

# Apply the frozen baseline to the same test rows
baseline_results = df.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "target_down",
        "baseline_action_score",
        "baseline_reason_code",
        "ctr"
    ]
].copy()

baseline_results = baseline_results.sort_values(
    ["baseline_action_score", "ctr"],
    ascending=[False, True]
).reset_index(drop=True)

baseline_p20 = baseline_results.head(20)["target_down"].mean()
baseline_p50 = baseline_results.head(50)["target_down"].mean()

print("Baseline results on the same test set")
print("-------------------------------------")
print(f"Precision@20: {baseline_p20:.3f}")
print(f"Precision@50: {baseline_p50:.3f}")

print("\nPositive pages selected:")
print(f"Top 20: {baseline_results.head(20)['target_down'].sum()}/20")
print(f"Top 50: {baseline_results.head(50)['target_down'].sum()}/50")

Baseline results on the same test set
-------------------------------------
Precision@20: 0.700
Precision@50: 0.640

Positive pages selected:
Top 20: 14/20
Top 50: 32/50


In [12]:
# Inspect the learned feature names and coefficients
feature_names = logistic_model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = logistic_model.named_steps[
    "model"
].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

importance = importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top 20 Logistic Regression features:")
print(
    importance[
        ["feature", "coefficient"]
    ].head(20).to_string(index=False)
)

Top 20 Logistic Regression features:
                       feature  coefficient
     num__impressions_last_30d   -35.339446
     num__impressions_prev_30d    28.889056
          num__impressions_90d     2.092414
      cat__position_tier_top_3    -1.297716
          num__clicks_prev_30d     1.092260
          num__clicks_last_30d    -0.866688
             num__sessions_90d     0.611044
                num__users_90d    -0.569541
    num__days_with_impressions     0.528176
        num__sessions_last_30d    -0.497426
      cat__freshness_tier_181+    -0.496616
         num__content_age_days    -0.470517
 cat__main_intent_navigational    -0.451279
            num__pageviews_90d     0.441163
   cat__position_tier_striking     0.390269
       cat__position_tier_deep     0.377007
       num__days_with_sessions    -0.367618
cat__word_count_tier_1000-2000     0.331968
      cat__impression_tier_low     0.312947
             num__avg_position    -0.252051


In [13]:
leakage_features = [
    "impressions_last_30d",
    "impressions_prev_30d"
]

safe_feature_columns = [
    feature for feature in feature_columns
    if feature not in leakage_features
]

print("Removed features:")
print(leakage_features)

print("\nOriginal feature count:", len(feature_columns))
print("Leakage-safe feature count:", len(safe_feature_columns))

print("\nRemaining recent-performance features:")
print([
    feature for feature in safe_feature_columns
    if "last_30d" in feature or "prev_30d" in feature
])

Removed features:
['impressions_last_30d', 'impressions_prev_30d']

Original feature count: 38
Leakage-safe feature count: 36

Remaining recent-performance features:
['clicks_last_30d', 'sessions_last_30d', 'clicks_prev_30d', 'sessions_prev_30d']


In [14]:
X_train_safe = df.iloc[train_idx][safe_feature_columns].copy()
X_test_safe = df.iloc[test_idx][safe_feature_columns].copy()

print("Leakage-safe training shape:", X_train_safe.shape)
print("Leakage-safe test shape:", X_test_safe.shape)
print("Leakage-safe features:", len(safe_feature_columns))

Leakage-safe training shape: (23837, 36)
Leakage-safe test shape: (6163, 36)
Leakage-safe features: 36


In [15]:
numeric_features_safe = X_train_safe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_safe = X_train_safe.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_transformer_safe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer_safe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_safe = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_safe, numeric_features_safe),
        ("cat", categorical_transformer_safe, categorical_features_safe)
    ]
)

print("Safe numeric features:", len(numeric_features_safe))
print("Safe categorical features:", len(categorical_features_safe))
print("Safe preprocessor created successfully.")

Safe numeric features: 27
Safe categorical features: 9
Safe preprocessor created successfully.


In [16]:
logistic_model_safe = Pipeline(
    steps=[
        ("preprocessor", preprocessor_safe),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

print("Leakage-safe Logistic Regression pipeline created successfully.")

Leakage-safe Logistic Regression pipeline created successfully.


In [17]:
logistic_model_safe.fit(X_train_safe, y_train)

print("Leakage-safe Logistic Regression trained successfully.")

Leakage-safe Logistic Regression trained successfully.


## 4. Errors and interpretation

The initial Logistic Regression achieved perfect Precision@20 and Precision@50, but feature inspection showed that it relied heavily on `impressions_last_30d` and `impressions_prev_30d`, which are directly related to how the proxy target was constructed.

I therefore retrained the same model after removing those two features. I will now evaluate the leakage-safe model on the same unseen test clients and inspect where it succeeds and fails.

In [18]:
safe_test_probabilities = logistic_model_safe.predict_proba(
    X_test_safe
)[:, 1]

safe_model_results = df.iloc[test_idx][
    ["content_id", "client_id"]
].copy()

safe_model_results["actual_down"] = y_test.values
safe_model_results["predicted_probability"] = safe_test_probabilities

safe_model_results = safe_model_results.sort_values(
    "predicted_probability",
    ascending=False
).reset_index(drop=True)

safe_model_results["rank"] = np.arange(
    1, len(safe_model_results) + 1
)

safe_p20 = safe_model_results.head(20)["actual_down"].mean()
safe_p50 = safe_model_results.head(50)["actual_down"].mean()

print("Leakage-safe Logistic Regression")
print("--------------------------------")
print(f"Precision@20: {safe_p20:.3f}")
print(f"Precision@50: {safe_p50:.3f}")

print("\nPositive pages selected:")
print(f"Top 20: {safe_model_results.head(20)['actual_down'].sum()}/20")
print(f"Top 50: {safe_model_results.head(50)['actual_down'].sum()}/50")

Leakage-safe Logistic Regression
--------------------------------
Precision@20: 0.850
Precision@50: 0.720

Positive pages selected:
Top 20: 17/20
Top 50: 36/50


### Error analysis

I will inspect the pages ranked highest by the leakage-safe model that were not actually labeled `down`. These false positives show where the model is recommending pages for review but the observed proxy label does not support the recommendation.

I will also compare the model's top-ranked pages with its missed `down` pages to understand where the ranking is strong and where it fails.

In [19]:
# False positives among the top 50 recommendations
top_50 = safe_model_results.head(50).copy()

false_positives = top_50[
    top_50["actual_down"] == 0
].copy()

print("False positives in top 50:", len(false_positives))
print("\nFalse-positive pages:")
print(
    false_positives[
        [
            "rank",
            "content_id",
            "client_id",
            "predicted_probability",
            "actual_down"
        ]
    ].to_string(index=False)
)

False positives in top 50: 14

False-positive pages:
 rank           content_id         client_id  predicted_probability  actual_down
   11 content_d896af65d5b6 client_4e07408562               0.961460            0
   18 content_a053262db2c0 client_4e07408562               0.913980            0
   20 content_7bf0aff635f4 client_4e07408562               0.898689            0
   21 content_f0d98be4b42c client_4e07408562               0.897828            0
   22 content_f4c93868660b client_4e07408562               0.896760            0
   24 content_41baf0722ad9 client_8527a891e2               0.893985            0
   26 content_7be5f150dc65 client_f369cb89fc               0.893096            0
   27 content_62c74ae619f8 client_4e07408562               0.892310            0
   29 content_374e795aab68 client_f369cb89fc               0.889145            0
   32 content_8725d0090607 client_4e07408562               0.886452            0
   39 content_ce59581533ca client_8527a891e2            

### False-positive patterns

The false positives are concentrated among a small number of clients. I will check whether the model's errors are disproportionately associated with particular clients before interpreting individual feature behavior.

In [20]:
false_positive_client_summary = (
    false_positives
    .groupby("client_id")
    .size()
    .sort_values(ascending=False)
)

print("False positives by client:")
print(false_positive_client_summary)

print("\nNumber of clients represented in false positives:")
print(false_positive_client_summary.shape[0])

False positives by client:
client_id
client_4e07408562    9
client_8527a891e2    3
client_f369cb89fc    2
dtype: int64

Number of clients represented in false positives:
3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.